# Verifying that the generated forms are positive maps

Every form `f` produced by `scripts/gen_pncp.jl` should define a **positive** map:
⟨x⊗y| f |x⊗y⟩ ≥ 0 for all product vectors |x⟩⊗|y⟩. This is the property the
generator targets, but it is only enforced approximately by the SOS relaxation,
so we check it here as quality control.

Two complementary checks (both in `common.jl`):

- `is_pos` — cheap random product-vector sampling; finds gross violations fast.
- `min_xy_form` — an alternating ("see-saw") SDP that converges to a local
  minimum of ⟨x⊗y| f |x⊗y⟩ from several restarts; a more reliable lower bound.

A value below `-tol` from either check means the form is **not** positive.

In [ ]:
include("common.jl")

In [ ]:
forms = load_forms("../pncp_4x4.jld2")
length(forms)

## Both checks on a single form

`is_pos` returns the smallest sampled value (and a witnessing x, y if it goes
negative); `min_xy_form` returns the see-saw minimum.

In [ ]:
f = forms[1]
random_min, _, _ = is_pos(f, 4, 4; attempts=1_000_000, tol=0.0)
seesaw_min, _, _ = min_xy_form(f, 4, 4; restarts=16, max_iter=40)
(random_min = random_min, seesaw_min = seesaw_min)

## Sweep over a sample of the library

The original notebook ran `min_xy_form` over all 10,000 forms (≈30 min) and wrote
the minima to a file. Here we sweep a subset and report the worst (most negative)
minimum seen — it should stay at or above `-tol`. Increase the range for a full
sweep.

In [ ]:
minima = Float64[]
@showprogress for f in forms[1:50]
    push!(minima, min_xy_form(f, 4, 4; restarts=8, max_iter=30)[1])
end
worst = minimum(minima)
println("worst see-saw minimum over sample: ", worst)
worst ≥ -1e-6 ? "all sampled forms positive" : "WARNING: a sampled form is not positive"